# HipAAsynth → Epic Seismometer — live cohort audit

This notebook loads a HipAAsynth synthetic cohort into **Seismometer** (Epic's open-source
AI-evaluation tool) and renders fairness / performance plots.

The Seismometer config package (`config.yml`, `usage_config.yml`, `dictionary.yml`,
`predictions.parquet`, `events.parquet`, `metadata.json`) is produced by
`seismometer_adapter.py` from our canonical `patients.json` + `results.csv`.
Point `SEIS_CONFIG_DIR` at that package (the `demo_seismometer.sh` runner sets it for you).

> ### ⚠️ What this notebook is — and is not
>
> - **`ModelScore` is a synthetic placeholder** generated by the adapter. HipAAsynth emits no model score; this column exists only so Seismometer has an output to evaluate.
> - **The AUROC / AUPRC are near-chance by construction and are NOT a performance result.** Do not read them as model quality.
> - **What this notebook demonstrates is the fairness and censoring pipeline:** whether sparse populations (e.g. *frontier*, *native*) survive Seismometer's `censor_min_count` gate and get scored as their own cohort, rather than being dropped from the audit.
>
> Seismometer is Epic's open-source tool; this notebook demonstrates *compatibility*, not a partnership or endorsement.


In [ ]:
import os, warnings, matplotlib
matplotlib.use("Agg")
warnings.filterwarnings("ignore")
from IPython.display import HTML as _IPHTML

# The adapter writes a self-contained package; Seismometer resolves data paths
# relative to the working directory, so we run from inside the package dir.
CONFIG_DIR = os.environ.get("SEIS_CONFIG_DIR", "seis_out")
os.chdir(CONFIG_DIR)
print("Seismometer config package:", os.getcwd())

import seismometer as sm
sm.run_startup(config_path=".")

from seismometer.seismogram import Seismogram
sg = Seismogram()

import json
with open("metadata.json") as f:
    thresholds = json.load(f)["thresholds"]

print(f"Loaded {len(sg.dataframe)} predictions")
print("censor_min_count threshold:", sg.censor_threshold)
print("Target:", sg.target, "| Score:", sg.output, "| Thresholds:", thresholds)
sm.cohort_list()

def render(obj):
    """Unwrap a Seismometer HTML/ipywidgets result to static HTML so it renders
    both in a live kernel and in headless `nbconvert --execute`."""
    for attr in ("value", "data"):
        if hasattr(obj, attr):
            return _IPHTML(getattr(obj, attr))
    return obj

## 1 · Overall model performance
ROC, precision–recall, calibration, and score histogram for the whole cohort.

In [ ]:
render(sm.plot_model_evaluation({}, sg.target, sg.output, thresholds))

*Caption — `ModelScore` is a synthetic placeholder; HipAAsynth emits no model score, so this AUROC/AUPRC is near-chance **by construction** and is not a performance result. This panel shows that the evaluation pipeline renders, not that a model is good.*


## 2 · Fairness across Rurality
The sparse-population axis (urban / suburban / rural / **frontier**) — the whole point of the audit.

In [ ]:
render(sm.plot_cohort_evaluation("Rurality", ["urban", "suburban", "rural", "frontier"],
                                 sg.target, sg.output, thresholds))

## 3 · Fairness table — Rurality × Race
Seismometer applies its own `censor_min_count` gate here; a cohort with too few observations is marked censored (❓) instead of scored.

In [ ]:
from seismometer.table.fairness import binary_metrics_fairness_table
from seismometer.data.performance import BinaryClassifierMetricGenerator

render(binary_metrics_fairness_table(
    BinaryClassifierMetricGenerator(),
    ["Accuracy", "Sensitivity", "Specificity", "PPV"],
    {"Rurality": ("urban", "suburban", "rural", "frontier"),
     "Race": ("white", "black", "hispanic", "native", "asian", "other")},
    0.25, sg.target, sg.output, 0.5))

## Interactive exploration (live Jupyter only)
When running this notebook in a live Jupyter kernel (not headless), uncomment below for
Seismometer's interactive widgets — pick cohorts and thresholds on the fly:

In [ ]:
# sm.ExploreModelEvaluation()
# sm.ExploreCohortEvaluation()
# sm.ExploreFairnessAudit()